# Figure S3 — data-processing and QC panels b and c only

This notebook does **not** redraw the QC panels from the manuscript caption. It locates an existing Oak notebook containing both original panel signatures and executes that notebook's plotting cells:

- **S3b:** total UMIs versus detected genes, colored by first-round barcode index;
- **S3c:** pre/post-QC distributions for total UMIs, detected genes, mitochondrial fraction, and ribosomal fraction.

The source notebook must contain the original scatter/violin code and the relevant QC variables. Generic `qc_panels.py` renderers are deliberately not used. If no unique source notebook is found, execution stops rather than synthesizing a replacement.


In [ ]:
from pathlib import Path
import json
import os
import sys
import pandas as pd

def _find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "cc_perturbseq").is_dir():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from within the "
        "cc-perturbseq-manuscript-analysis checkout."
    )

REPO_ROOT = _find_repo_root()
for path in (REPO_ROOT, REPO_ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from src.cc_perturbseq.figure_notebook import (
    initialize_figure,
    panel_directory,
    write_receipt,
)
from src.cc_perturbseq.source_notebook_runner import (
    discover_notebook,
    execute_notebook,
    prepare_s3_source_reproduction,
)

FIGURE = {
    "figure_id": "Figure_S03",
    "title": "Figure S3 — CC-Perturb-seq data processing and quality control",
    "scope": "Panels b and c only, executed from an existing source notebook",
    "manuscript": "Working CC-Perturb-seq EC Differentiation - Text V3",
    "panels": [
        {
            "panel": "b",
            "description": (
                "Total UMIs versus detected genes colored by Round-1 barcode"
            ),
            "type": "source_notebook",
            "required_inputs": [],
            "candidate_sources": ["Existing preprocessing/QC notebook on Oak"],
        },
        {
            "panel": "c",
            "description": (
                "Pre/post-QC distributions for counts, genes, mitochondrial "
                "fraction and ribosomal fraction"
            ),
            "type": "source_notebook",
            "required_inputs": [],
            "candidate_sources": ["Existing preprocessing/QC notebook on Oak"],
        },
    ],
}
ctx = initialize_figure(FIGURE)
OUTDIR = panel_directory(ctx, "b-c")


In [ ]:
SOURCE_ENV = "CCPERTURBSEQ_FIGURE_S03_SOURCE_NOTEBOOK"
explicit_source = os.environ.get(SOURCE_ENV)
search_roots = [
    Path(
        "/oak/stanford/groups/engreitz/Users/opushkar/cc_perturb/"
        "2.preprocessing_tonys_pipeline"
    ),
    Path("/oak/stanford/groups/engreitz/Users/opushkar/cc_perturb"),
    Path("/oak/stanford/groups/engreitz/Users/tonyzeng"),
    REPO_ROOT / "recovery",
]

SOURCE_NOTEBOOK, candidate_rows = discover_notebook(
    explicit_path=explicit_source,
    search_roots=search_roots,
    required_token_groups=[
        ("total_counts", "total_umis", "ncount_rna"),
        ("n_genes_by_counts", "detected_genes", "nfeature_rna"),
        ("pct_counts_mt", "mitochondrial_fraction"),
        ("pct_counts_ribo", "ribosomal_fraction"),
        (
            "round1",
            "round_1",
            "round 1",
            "bci",
            "barcode index",
            "first-round barcode",
        ),
        ("scatter(", "geom_point(", "sns.scatterplot"),
        ("violinplot", "geom_violin", "violin("),
    ],
    preferred_tokens=[
        "median_abs_deviation",
        "median absolute deviation",
        "qc_pass",
        "before",
        "after",
        "pct_counts_ribo",
        "round1",
        "barcode index",
        "d0",
        "d1",
        "d2",
        "d3",
    ],
    excluded_tokens=[
        "250730_etn10_parsev3_mixedipsc_k562",
        "hippo_timecourse",
    ],
)
candidate_table = pd.DataFrame(candidate_rows)
candidate_path = OUTDIR / "Figure_S03_source_notebook_candidates.csv"
candidate_table.to_csv(candidate_path, index=False)
display({
    "selected_source_notebook": str(SOURCE_NOTEBOOK),
    "explicit_source_environment_variable": SOURCE_ENV,
    "candidate_report": str(candidate_path),
})
display(candidate_table.head(20))


In [ ]:
PREPARED_NOTEBOOK = OUTDIR / "Figure_S03bc_source_prepared.ipynb"
EXECUTED_NOTEBOOK = OUTDIR / "Figure_S03bc_source_executed.ipynb"

prepared, provenance = prepare_s3_source_reproduction(
    SOURCE_NOTEBOOK,
    PREPARED_NOTEBOOK,
    capture_output_dir=OUTDIR,
)
provenance["candidate_report"] = str(candidate_path)
provenance_path = OUTDIR / "Figure_S03bc_source_provenance.json"
provenance_path.write_text(
    json.dumps(provenance, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
display(provenance)


In [ ]:
execute_notebook(
    prepared,
    EXECUTED_NOTEBOOK,
    working_directory=SOURCE_NOTEBOOK.parent,
    timeout=14_400,
)

manifest_path = OUTDIR / "Figure_S03_source_capture_manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError(
        "The executed source notebook did not produce the capture manifest: "
        f"{manifest_path}"
    )
capture_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
counts = capture_manifest.get("capture_counts", {})
if int(counts.get("b", 0)) < 1 or int(counts.get("c", 0)) < 1:
    raise RuntimeError(
        "The source execution did not capture at least one original figure for "
        f"both S3b and S3c: {counts}"
    )

write_receipt(
    ctx,
    "b-c",
    status="EXISTING_SOURCE_NOTEBOOK_CELLS_EXECUTED",
    inputs={
        "source_notebook": str(SOURCE_NOTEBOOK),
        "source_sha256": provenance["source_sha256"],
    },
    outputs={
        "candidate_report": str(candidate_path),
        "prepared_source_notebook": str(PREPARED_NOTEBOOK),
        "executed_source_notebook": str(EXECUTED_NOTEBOOK),
        "provenance": str(provenance_path),
        "capture_manifest": str(manifest_path),
        **{
            f"captured_{index:03d}": path
            for index, path in enumerate(
                capture_manifest.get("captured_files", []), start=1
            )
        },
    },
    parameters={
        "panels": ["b", "c"],
        "generic_qc_renderer_used": False,
        "source_cell_matches": provenance["matches"],
        "source_selection": (
            "explicit environment variable"
            if explicit_source
            else "strict signature discovery with unique-best requirement"
        ),
    },
)
display(capture_manifest)
